# 🧠 Face Stats — Feature Extraction Notebook

This notebook extracts **classical beauty metrics** from each of the
preprocessed 512×512 aligned faces.

These features include:

- Facial symmetry score  
- Geometric ratios (eyes–nose–mouth distances)  
- Golden-ratio proportionality features  
- Eye alignment, eye size ratio, eye spacing  
- Jawline width ratio  
- Facial thirds proportions  
- Texture smoothness  
- Skin color evenness  
- Contrast uniformity  
- Sharpness metrics  
- Lighting symmetry  

All features will be stored in:
- ```embeddings/features.npy```
- ```embeddings/feature_index.json```

These become the structured inputs for the attractiveness model.


In [1]:
import os
import cv2
import json
import glob
import numpy as np
import mediapipe as mp
from tqdm import tqdm
from PIL import Image

# Paths
DATA_DIR = "dataset_preprocessed"
META_DIR = "dataset_metadata"
OUT_DIR = "embeddings"
os.makedirs(OUT_DIR, exist_ok=True)

mp_face = mp.solutions.face_mesh.FaceMesh(
    static_image_mode=True,
    refine_landmarks=True,
    max_num_faces=1,
    min_detection_confidence=0.5
)

I0000 00:00:1763294992.989597 3822251 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M1


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1763294993.010867 3822349 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1763294993.020447 3822349 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


# 🧩 Helper Functions

This section defines reusable functions:

- load_image  
- compute_symmetry  
- compute_texture_features  
- compute_color_uniformity  
- compute_geometric_ratios  
- extract_landmark_array  

These functions operate on a single 512×512 preprocessed image.

In [2]:
def load_image(path):
    img = cv2.imread(path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img, img_rgb

def get_landmarks(img_rgb):
    res = mp_face.process(img_rgb)
    if not res.multi_face_landmarks:
        return None
    lms = res.multi_face_landmarks[0].landmark
    pts = np.array([[lm.x, lm.y] for lm in lms])
    return pts


# 🔍 Facial Symmetry Feature

We compute symmetry by comparing the left and right halves of the face:

- Split at the vertical midpoint  
- Flip left half  
- Compute structural similarity via pixel difference  

Produces a number in **[0, 1]**, where 1 = perfect symmetry.

In [3]:
def symmetry_score(img_rgb):
    h, w, _ = img_rgb.shape
    mid = w // 2
    
    left = img_rgb[:, :mid]
    right = img_rgb[:, mid:]
    right_flipped = np.fliplr(right)
    
    # Pad if shapes mismatch
    min_w = min(left.shape[1], right_flipped.shape[1])
    left = left[:, :min_w]
    right_flipped = right_flipped[:, :min_w]
    
    diff = np.abs(left.astype("float") - right_flipped.astype("float"))
    score = 1.0 - (diff.mean() / 255.0)
    return float(max(0.0, min(1.0, score)))

# 🎨 Texture & Smoothness Features

We compute:

- Local variance (skin smoothness)
- Laplacian sharpness (edge clarity)
- Gradient magnitude (texture energy)

These help quantify:
- Age cues  
- Skin clarity  
- Image quality  


In [4]:
def texture_features(img_rgb):
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    
    # Local variance (smoothness)
    smooth = gray.astype("float")
    local_var = smooth.std()

    # Sharpness via Laplacian
    lap = cv2.Laplacian(gray, cv2.CV_64F)
    sharpness = np.mean(np.abs(lap))

    # Gradient magnitude
    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0)
    gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1)
    grad_mag = np.mean(np.sqrt(gx**2 + gy**2))

    return float(local_var), float(sharpness), float(grad_mag)

# 🌈 Color Uniformity (Skin Evenness)

We compute standard deviation across:

- Hue  
- Saturation  
- Value  

Uniform skin coloring correlates with perceived attractiveness.

In [5]:
def color_uniformity(img_rgb):
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    h_std = hsv[:,:,0].std()
    s_std = hsv[:,:,1].std()
    v_std = hsv[:,:,2].std()
    return float(h_std), float(s_std), float(v_std)

# 📐 Geometric Ratios from Landmarks

Using the MediaPipe 468-landmark mesh, we compute:

- Eye size ratio  
- Eye spacing ratio  
- Nose length to face height  
- Mouth width to face width  
- Facial thirds ratio (hairline–brow–nose–chin)  
- Jaw width ratio  

In [6]:
def geometric_features(pts):
    # Face width = distance between left cheek (234) & right cheek (454)
    face_width = np.linalg.norm(pts[234] - pts[454])

    # Face height = distance between chin (152) & forehead/glabella (8)
    face_height = np.linalg.norm(pts[152] - pts[8])

    # Eye size ratio (big eyes effect)
    left_eye_w = np.linalg.norm(pts[33] - pts[133])
    right_eye_w = np.linalg.norm(pts[362] - pts[263])
    eye_size_ratio = (left_eye_w + right_eye_w) / (2 * face_width)

    # Eye spacing (golden-features)
    eye_spacing = np.linalg.norm(pts[133] - pts[362]) / face_width

    # Nose metrics
    nose_length = np.linalg.norm(pts[1] - pts[2])
    nose_ratio = nose_length / face_height

    # Mouth width
    mouth = np.linalg.norm(pts[61] - pts[291]) / face_width

    # Facial thirds (8→168, 168→2, 2→152)
    upper = np.linalg.norm(pts[8] - pts[168])
    middle = np.linalg.norm(pts[168] - pts[2])
    lower = np.linalg.norm(pts[2] - pts[152])
    thirds_ratio = np.array([upper, middle, lower]) / face_height

    return np.array([
        face_width, face_height,
        eye_size_ratio, eye_spacing,
        nose_ratio, mouth,
        thirds_ratio[0], thirds_ratio[1], thirds_ratio[2]
    ], dtype=float)

# 🚀 Batch Feature Extraction

This will:

- Loop over all preprocessed images  
- Extract landmarks  
- Compute all features  
- Save to `embeddings/features.npy`  
- Save feature index to JSON  

Expected time:
- ~15–30 minutes on M1  
- ~1 hour on slower machines  

In [7]:
files = sorted(glob.glob(f"{DATA_DIR}/*.png"))
all_features = []
failed = []

for path in tqdm(files, desc="Extracting features"):
    img, img_rgb = load_image(path)
    pts = get_landmarks(img_rgb)

    if pts is None:
        failed.append(path)
        continue

    sym = symmetry_score(img_rgb)
    t_var, sharp, grad = texture_features(img_rgb)
    h_std, s_std, v_std = color_uniformity(img_rgb)
    geom = geometric_features(pts)

    feature_vec = np.concatenate([
        np.array([sym, t_var, sharp, grad, h_std, s_std, v_std]),
        geom
    ])

    all_features.append(feature_vec)

all_features = np.array(all_features)
np.save(os.path.join(OUT_DIR, "features.npy"), all_features)

with open(os.path.join(OUT_DIR, "feature_index.json"), "w") as f:
    json.dump({
        "features": [
            "symmetry",
            "texture_variance",
            "sharpness",
            "gradient_magnitude",
            "hue_std",
            "sat_std",
            "val_std",
            "face_width",
            "face_height",
            "eye_size_ratio",
            "eye_spacing",
            "nose_ratio",
            "mouth_ratio",
            "thirds_upper",
            "thirds_middle",
            "thirds_lower"
        ],
        "count": len(all_features)
    }, f)

print("Done. Saved features to embeddings/features.npy")
print("Failures:", len(failed))


Extracting features: 100%|██████████| 125686/125686 [38:49<00:00, 53.94it/s] 


Done. Saved features to embeddings/features.npy
Failures: 238


# 🎉 Next Steps

Now that we have:

- `embeddings/features.npy`
- `embeddings/feature_index.json`

We are ready for:

### 1. Embedding Extraction Notebook (CLIP / Aesthetic Embeddings)
### 2. Labeling Pipeline (auto-label or human-in-loop)
### 3. Attractiveness Model Training
### 4. PCA + Visualizations
### 5. Composite Faces per Decile